## Installing Dependencies

RAG pipline:
docunment loading 
embedding model 
LLM model 
vector database 


In [48]:
print("Installing Dependencies\n");

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken
print('Installation completed');


Installing Dependencies

Installation completed



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
LLM_Provider = 'groq'
LLM_Model = 'openai/gpt-oss-20b'



In [50]:
import os
from dotenv import load_dotenv  
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("GROQ API Key is Imported.")
else:
    print("GROQ API Key is not found")

GROQ API Key is Imported.


In [51]:
from langchain_community.document_loaders import PyPDFDirectoryLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
CORPUS_PATH = "./zyro-dynamics-hr-corpus/" 
loader = PyPDFDirectoryLoader(CORPUS_PATH) 
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 39 documents


In [52]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 107 chunks


## Embeddings
HuggingFaceEmbeddings

In [53]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)
print("Embedding Done...!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1687.78it/s]


Embedding Done...!


## VectorDB + Retrival 
FAISS VectorDB 

In [54]:
from langchain_community.vectorstores import FAISS 
vectorstore = FAISS.from_documents( chunks , embedding_model)

#retriever 

retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) 
print("Done..!")

Done..!


## LLM Initialization 
LLM - groq 

In [55]:
## LLM Initialization

from langchain_groq import ChatGroq 
llm_model= ChatGroq( 
model = LLM_Model,
temperature = 0.7,
max_tokens = 500 
)
print(f"LLM Model {LLM_Model} initialized")

LLM Model openai/gpt-oss-20b initialized


## Rag Chain 

In [56]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 
from langsmith import traceable 
RAG_PROMPT =ChatPromptTemplate.from_template(
    """" You Are the HR Assistant Chatbot for zyro dynamcis you are assigned to answer the queations which are related to HR policies.
      you are suppose to answer this questions with the help of the context data which is provided to you in the documents. 
      Use it accordingly and answer the questions if you haven't found the related information in the context say 
      "I am Sorry currently I am not able to answer your question i will work and answer you question 
      context:{context},
      question:{question} """
)

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)

@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer =chain.invoke({"context" : context, "question" : question})
    return {"answer" : answer } 


## Guardials 
To ensure that no context is used beyond the corpus 

In [58]:
GUARDIAL_PROMPT =ChatPromptTemplate.from_template(  """
You are a smart HR assistance chatbot . you are assigned to answer the questions related to the HR policies so decide wheather the question is rlated to HR policies and answer or something else which is out of context you have 
Respond with one word :IN_SCOPE or NOT_IN_SCOPE.
question:{question} 
""") 

REFUSAL_MESSAGE = (
    "I'm an HR assistance chatbot and can answer question related to HR policies." 
    "Sorry..! I don't have information to answer your question if you have any other question related to HR policies please ask me"
)

def ask_bot(question:str):
    guardial_chain =GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict  = guardial_chain.invoke({"question": question}).strip().upper()   
    if 'NOT_IN_SCOPE' in verdict:
        return {'answer' : REFUSAL_MESSAGE, "sources" : [] }

    rag_result = rag_chain(question)
    return {
        "answer": rag_result.get("answer", ""),
        "sources": rag_result.get("sources", [])
    }
print("Gaurdial Initialized..!")


Gaurdial Initialized..!


In [59]:
test_questions = [
    "How many casual leaves do I get per year?", 
    "What is the internet reimbursement limit?", 
    "What's the capital of France?", 
]
for i, q in enumerate(test_questions,1):
    result = ask_bot(q)
    print(f"Q{i} : {q}")
    print(f"A{i} : {result['answer']}")
    if result["sources"]:
        print(
            f"Sources: "
            f"{[ d.metadata.get('source') for d in result ['sources']]}"  
        )
    print("-" * 60)

Q1 : How many casual leaves do I get per year?
A1 : I am Sorry currently I am not able to answer your question i will work and answer you question
------------------------------------------------------------
Q2 : What is the internet reimbursement limit?
A2 : The internet reimbursement limit is **Rs. 1,000 per month** for eligible Full Remote employees at Level 5 and above.
------------------------------------------------------------
Q3 : What's the capital of France?
A3 : I'm an HR assistance chatbot and can answer question related to HR policies.Sorry..! I don't have information to answer your question if you have any other question related to HR policies please ask me
------------------------------------------------------------


In [ ]:
import pandas as pd
test_df = pd.read_csv("test.csv")
test_df
answers = []
for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers

submission = test_df[["question_id", "answer"]]
submission.to_csv("submission.csv", index=False)
submission.head()



[1/20] How does my Earned Leave accrue every month?
-> Earned Leave (EL) is added to your balance on a monthly basis based on your service period:

| Status | Accrual Rate | Availability |
|--------|--------------|--------------|
| **After 1 year of continuous service** (with at least 240 days worked that year) | **1.25 days per month** | Available for use immediately (subject to manager approval) |
| **During probation** | **0.5 days per month** | Accumulates, but can only be used once you’ve confirmed your probation period |

So, for every month you’re employed after the first year, your EL balance increases by 1.25 days. If you’re still in probation, you’ll add 0.5 days each month until your probation is confirmed.
------------------------------------------------------------
[2/20] How much Earned Leave can I carry forward to next year?
-> I am sorry, currently I am not able to answer your question. I will work and answer you question.
-----------------------------------------------